# String and Regex Metrics

The simplest evaluation metrics check whether your program's output matches a reference string. This notebook walks through four flavors of string-based metrics in DSPy: exact match, regex matching (for blog-style structural checks and keyword inclusion), and Levenshtein edit distance for fuzzier character-level comparisons. These metrics are deterministic, free to run, and a great starting point before you reach for embeddings or LLM judges.

In [ ]:
%pip install -r ../requirements.txt -q

In [ ]:
import dspy
from dotenv import load_dotenv

load_dotenv()

dspy.configure(lm=dspy.LM('openai/gpt-5-mini'))

## Exact match

When there is a single correct answer, the easiest metric is to compare the gold reference field to the predicted field directly. Most DSPy metric functions take an `example`, a `pred`, and an optional `trace` argument.

In [ ]:
def exact_match(gold: dspy.Example, pred: dspy.Prediction, trace=None):
    """
    A simple exact match metric that compares the 'answer' field of the
    gold example and the prediction.
    """
    return gold.answer == pred.answer


example = dspy.Example(answer="Paris")
pred = dspy.Prediction(answer="Paris")

print(exact_match(example, pred))

DSPy also ships built-in convenience metrics that normalize punctuation, special characters, and whitespace before comparing:

- `dspy.evaluate.metrics.answer_exact_match` checks the final answer.
- `dspy.evaluate.metrics.answer_passage_match` checks whether the correct answer appears anywhere in `pred.context` (useful for RAG retrieval).

Exact match is strict — `"The capital is Paris."` will fail against gold `"Paris"`. For fuzzier comparisons keep reading.

## Regex: blog-post structural checks

When the output is creative (a blog post, an email) you can't pin down the exact text, but you can still enforce structure. Here's a metric that checks the generated blog post has at least three Markdown H2 headers.

In [ ]:
import re

def blog_structure_metric(example, pred, trace=None):
    """Check if the blog post has at least 3 Markdown headers."""

    # Regex pattern for H2 headers (lines starting with ##)
    # (?m) flag enables multiline matching so ^ matches start of line
    header_pattern = r'(?m)^##\s+.+'

    # Find all headers in the generated answer
    headers = re.findall(header_pattern, pred.answer)

    # Metric passes if we found at least 3 headers
    return len(headers) >= 3


blog_post = """Here is your post:

## Introduction
DSPy is great...

## How it Works
It uses optimizers...

## Conclusion
You should try it.
"""

example = dspy.Example(topic="DSPy")
pred = dspy.Prediction(answer=blog_post)

print(blog_structure_metric(example, pred))

## Regex: keyword inclusion

Another common use of regex is verifying that important keywords or entities appear in the answer, regardless of phrasing. Word boundaries (`\b`) keep `"car"` from matching inside `"carbon"`.

In [ ]:
def keyword_inclusion_metric(example, pred, trace=None):
    # example.keywords is a list, e.g. ["chlorophyll", "sunlight", "energy"]
    text = pred.answer.lower()

    for keyword in example.keywords:
        # Whole-word match via \b boundary
        pattern = r'\b' + re.escape(keyword.lower()) + r'\b'

        if not re.search(pattern, text):
            return False  # Failed if any keyword is missing

    return True


example = dspy.Example(keywords=["chlorophyll", "sunlight", "energy"])
pred = dspy.Prediction(
    answer="Plants use chlorophyll to capture sunlight and convert it into energy."
)

print(keyword_inclusion_metric(example, pred))

### Partial-credit variant

All-or-nothing metrics make optimizers brittle. A small change can flip the score from 0 to 1 with no smooth gradient in between. A partial-credit version returns a float between 0.0 and 1.0:

In [ ]:
def keyword_inclusion_metric(example, pred, trace=None):
    # e.g. example.keywords = ["chlorophyll", "sunlight", "energy"]
    text = pred.answer.lower()
    matches = 0

    for keyword in example.keywords:
        pattern = r'\b' + re.escape(keyword.lower()) + r'\b'

        if re.search(pattern, text):
            matches += 1

    # Return score between 0.0 and 1.0
    return matches / len(example.keywords)


pred = dspy.Prediction(
    answer="Plants use chlorophyll to capture sunlight."  # missing 'energy'
)

print(keyword_inclusion_metric(example, pred))  # 2 / 3 = 0.666...

## Edit distance (Levenshtein)

Edit distance counts how many single-character insertions, deletions, or substitutions are required to turn one string into another. It's useful for OCR outputs, spelling correction, or any task where small character-level errors should still earn credit.

In [ ]:
from Levenshtein import distance

# 1. Define your metric
def edit_distance_metric(example, pred, trace=None, threshold=3):
    gold = str(example.answer).strip().lower()
    predicted = str(pred.answer).strip().lower()
    dist = distance(gold, predicted)
    return dist <= threshold

# 2. Create the DSPy objects
# dspy.Example represents the "ground truth" from your dataset
example = dspy.Example(answer="Barack Obama")

# dspy.Prediction represents the output from your model
pred = dspy.Prediction(answer="Barrack Obama")

# 3. Test the metric
result = edit_distance_metric(example, pred)

print(f"Passes metric? {result}")  # True
print(f"Raw distance: {distance('barack obama', 'barrack obama')}")  # 1

Edit distance ignores meaning — `"good"` and `"great"` are 3 edits apart, while `"good"` and `"goof"` are only 1. For creative or long-form outputs where meaning matters, jump to the semantic-similarity notebook next.